<a href="https://colab.research.google.com/github/Pensive1881/DSR44_2025/blob/main/jaguar_semantic_segmentation_CVAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Define your username to get your appropriate data split into CVAT

In [ ]:
# Use lower case for your name
MY_NAME = "antonio"

## Install FiftyOne

In [ ]:
%%capture
!uv pip install fiftyone==1.9.0

## Get CVAT credentials

This needs to run before importing FiftyOne.

In [ ]:
import os
from google.colab import userdata
os.environ['FIFTYONE_CVAT_PASSWORD'] = userdata.get('FIFTYONE_CVAT_PASSWORD')
os.environ['FIFTYONE_CVAT_USERNAME'] = userdata.get('FIFTYONE_CVAT_USERNAME')


## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/gdrive')
%cd /gdrive

Mounted at /gdrive
/gdrive


Add this folder to your Google Drive through Organize -> Add Shortcut
https://drive.google.com/drive/folders/1whhJD0tn0NLOitKLtt2BeJxMYukfITzg?usp=drive_link

In [ ]:
from pathlib import Path
train_set_files = Path('/gdrive/MyDrive/subset_train_set_parquet_files')
os.listdir(train_set_files)

['cropped_body-00000-of-00005.parquet']

## Create the FiftyOne dataset

In [ ]:
"""## Extract images from Parquet files"""

import pyarrow.parquet as pq
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import fiftyone as fo

def extract_images_from_parquet(parquet_dir, image_type=None, output_dir="fiftyone_images"):
    """
    Extract images from Parquet files and save to disk.

    Args:
        parquet_dir: Directory containing Parquet files
        image_type: Specific image type to extract (cropped_body, cropped_head, segmented_body) or None for all
        output_dir: Directory to save extracted images

    Returns:
        List of tuples: (image_path, label, image_type, original_filename)
    """
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True)

    # Get relevant parquet files
    parquet_files = sorted(Path(parquet_dir).glob("*.parquet"))

    if image_type:
        parquet_files = [f for f in parquet_files if f.name.startswith(image_type)]
        if not parquet_files:
            raise ValueError(f"No files found for image type: {image_type}")

    print(f"Found {len(parquet_files)} parquet files to process")

    samples_data = []

    for parquet_file in parquet_files:
        # Extract image type from filename (e.g., "cropped_body-00000-of-00005.parquet")
        current_image_type = parquet_file.name.split('-')[0]

        print(f"\nProcessing: {parquet_file.name}")
        table = pq.read_table(parquet_file)

        # Create subdirectory for this image type
        type_dir = output_path / current_image_type
        type_dir.mkdir(exist_ok=True)

        # Extract data
        filenames = table.column('filename').to_pylist()
        labels = table.column('label').to_pylist()
        images = table.column('image').to_pylist()

        for idx, (filename, label, img_data) in enumerate(tqdm(
            zip(filenames, labels, images),
            total=len(filenames),
            desc=f"Extracting {current_image_type}"
        )):
            if img_data and img_data['bytes']:
                # Create unique filename: label_originalname_imagetype
                base_name = Path(filename).stem
                ext = Path(filename).suffix or '.jpg'
                unique_name = f"{label}_{base_name}_{current_image_type}{ext}"
                img_path = type_dir / unique_name

                # Save image
                try:
                    img = Image.open(BytesIO(img_data['bytes']))
                    img.save(img_path)

                    samples_data.append({
                        'filepath': str(img_path.absolute()),
                        'label': label,
                        'image_type': current_image_type,
                        'original_filename': filename
                    })
                except Exception as e:
                    print(f"Warning: Failed to process {filename}: {e}")

    return samples_data

"""## Create FiftyOne Dataset"""

def create_fiftyone_dataset(samples_data, dataset_name="jaguar_reid", persistent=True):
    """
    Create a FiftyOne dataset from extracted samples.

    Args:
        samples_data: List of sample dictionaries with filepath, label, image_type, original_filename
        dataset_name: Name for the FiftyOne dataset
        persistent: Whether to make the dataset persistent

    Returns:
        FiftyOne Dataset object
    """
    # Delete existing dataset with same name
    if dataset_name in fo.list_datasets():
        print(f"Deleting existing dataset: {dataset_name}")
        fo.delete_dataset(dataset_name)

    # Create new dataset
    dataset = fo.Dataset(dataset_name, persistent=persistent)

    print(f"\nCreating FiftyOne dataset: {dataset_name}")

    # Create samples
    samples = []
    for data in tqdm(samples_data, desc="Creating FiftyOne samples"):
        sample = fo.Sample(filepath=data['filepath'])

        # Add classification label
        sample['ground_truth'] = fo.Classification(label=data['label'])

        # Add custom fields
        sample['image_type'] = data['image_type']
        sample['original_filename'] = data['original_filename']

        samples.append(sample)

    # Add samples to dataset
    dataset.add_samples(samples)

    print(f"\nDataset created successfully!")
    print(f"  Name: {dataset_name}")
    print(f"  Total samples: {len(dataset)}")
    print(f"  Unique labels: {len(dataset.distinct('ground_truth.label'))}")
    print(f"  Image types: {dataset.distinct('image_type')}")

    return dataset

"""## Run the pipeline"""

# Configuration
PARQUET_DIR = train_set_files
OUTPUT_DIR = "/gdrive/MyDrive/fiftyone_images"  # Save in Google Drive
DATASET_NAME = "jaguar_reid"
IMAGE_TYPE = None  # None for all types, or specify: 'cropped_body', 'cropped_head', 'segmented_body'

# Step 1: Extract images from Parquet files
print("Step 1: Extracting images from Parquet files...")
samples_data = extract_images_from_parquet(
    PARQUET_DIR,
    image_type=IMAGE_TYPE,
    output_dir=OUTPUT_DIR
)

print(f"\nExtracted {len(samples_data)} images")

# Step 2: Create FiftyOne dataset
print("\nStep 2: Creating FiftyOne dataset...")
dataset = create_fiftyone_dataset(
    samples_data,
    dataset_name=DATASET_NAME,
    persistent=True
)

# Print summary statistics
print("\n" + "="*60)
print("Dataset Summary:")
print("="*60)
print(dataset)
print("\nLabel distribution:")
for label in sorted(dataset.distinct('ground_truth.label')):
    count = len(dataset.match(fo.ViewField('ground_truth.label') == label))
    print(f"  {label}: {count} samples")

"""## Launch FiftyOne App

To view the dataset in Google Colab, use the remote session:
"""

# Launch FiftyOne app with remote session for Colab
session = fo.launch_app(dataset, auto=False)

# The session will provide a link you can click to view the dataset
print("\nClick the link above to view your dataset in FiftyOne!")

print(session.url)

/usr/local/lib/python3.12/dist-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'


Step 1: Extracting images from Parquet files...
Found 1 parquet files to process

Processing: cropped_body-00000-of-00005.parquet


Extracting cropped_body: 100%|██████████| 620/620 [10:11<00:00,  1.01it/s]



Extracted 620 images

Step 2: Creating FiftyOne dataset...
You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information



Creating FiftyOne dataset: jaguar_reid


...


Dataset created successfully!
  Name: jaguar_reid
  Total samples: 620
  Unique labels: 4
  Image types: ['cropped_body']

Dataset Summary:
Name:        jaguar_reid
Media type:  image
Num samples: 620
Persistent:  True
Tags:        []
Sample fields:
    id:                fiftyone.core.fields.ObjectIdField
    filepath:          fiftyone.core.fields.StringField
    tags:              fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:          fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:        fiftyone.core.fields.DateTimeField
    last_modified_at:  fiftyone.core.fields.DateTimeField
    ground_truth:      fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Classification)
    image_type:        fiftyone.core.fields.StringField
    original_filename: fiftyone.core.fields.StringField

Label distribution:
  Alira: 45 samples
  Bororo: 47 samples
  Marcela: 237 samples
  Ousado: 291 samples
Sessi

INFO:fiftyone.core.session.session:Session launched. Run `session.show()` to open the App in a cell output.



Welcome to

███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.9.0

If you're finding FiftyOne helpful, here's how you can get involved:

|
|  ⭐⭐⭐ Give the project a star on GitHub ⭐⭐⭐
|  https://github.com/voxel51/fiftyone
|
|  🚀🚀🚀 Join the FiftyOne Discord community 🚀🚀🚀
|  https://community.voxel51.com/
|



INFO:fiftyone.core.session.session:
Welcome to

███████╗██╗███████╗████████╗██╗   ██╗ ██████╗ ███╗   ██╗███████╗
██╔════╝██║██╔════╝╚══██╔══╝╚██╗ ██╔╝██╔═══██╗████╗  ██║██╔════╝
█████╗  ██║█████╗     ██║    ╚████╔╝ ██║   ██║██╔██╗ ██║█████╗
██╔══╝  ██║██╔══╝     ██║     ╚██╔╝  ██║   ██║██║╚██╗██║██╔══╝
██║     ██║██║        ██║      ██║   ╚██████╔╝██║ ╚████║███████╗
╚═╝     ╚═╝╚═╝        ╚═╝      ╚═╝    ╚═════╝ ╚═╝  ╚═══╝╚══════╝ v1.9.0

If you're finding FiftyOne helpful, here's how you can get involved:

|
|  ⭐⭐⭐ Give the project a star on GitHub ⭐⭐⭐
|  https://github.com/voxel51/fiftyone
|
|  🚀🚀🚀 Join the FiftyOne Discord community 🚀🚀🚀
|  https://community.voxel51.com/
|




Click the link above to view your dataset in FiftyOne!
https://5151-m-hm-537h452vexbx-b.asia-east1-0.prod.colab.dev?polling=true


## Split the FiftyOne dataset on six parts

Split the FO dataset into six parts, each assigned to an user:

* Antonio
* Juan
* Thede
* Uma
* Alex
* Rami



In [ ]:
"""## Split the FiftyOne dataset into six parts

Split the dataset into six roughly equal parts, one for each annotator.
Each split will be deterministic based on sample order and alphabetic user order.
"""

import fiftyone as fo

# Load the original dataset
dataset = fo.load_dataset("jaguar_reid")

# Define the users in alphabetical order
users = sorted(["antonio", "juan", "thede", "uma", "alex", "rami"])
print(f"Users in alphabetical order: {users}")

# Get unique labels to ensure balanced distribution
unique_labels = sorted(dataset.distinct('ground_truth.label'))
print(f"\nFound {len(unique_labels)} unique labels")
print(f"Total samples: {len(dataset)}")

# Strategy: Split samples per label to maintain class balance across users
# Samples are assigned in order without shuffling
samples_per_user = {user: [] for user in users}

for label in unique_labels:
    # Get all samples for this label (in dataset order)
    label_view = dataset.match(fo.ViewField('ground_truth.label') == label)
    label_samples = list(label_view)

    # Distribute samples across users in order
    samples_per_label = len(label_samples)
    samples_per_user_count = samples_per_label // len(users)

    for i, user in enumerate(users):
        start_idx = i * samples_per_user_count
        # Last user gets any remaining samples
        end_idx = start_idx + samples_per_user_count if i < len(users) - 1 else samples_per_label
        samples_per_user[user].extend(label_samples[start_idx:end_idx])

# Create separate datasets for each user
user_datasets = {}

for user in users:
    dataset_name = f"jaguar_reid_{user.lower()}"

    # Delete existing dataset if it exists
    if dataset_name in fo.list_datasets():
        print(f"Deleting existing dataset: {dataset_name}")
        fo.delete_dataset(dataset_name)

    # Create new dataset for this user
    user_dataset = fo.Dataset(dataset_name, persistent=True)
    user_dataset.add_samples(samples_per_user[user])

    # Copy mask targets from original dataset
    user_dataset.default_mask_targets = dataset.default_mask_targets
    user_dataset.save()

    user_datasets[user] = user_dataset

    print(f"\n{user}'s dataset: {dataset_name}")
    print(f"  Samples: {len(user_dataset)}")
    print(f"  Labels: {len(user_dataset.distinct('ground_truth.label'))}")

# Print summary
print("\n" + "="*60)
print("Dataset Split Summary:")
print("="*60)
for user, user_dataset in user_datasets.items():
    label_dist = {}
    for label in user_dataset.distinct('ground_truth.label'):
        count = len(user_dataset.match(fo.ViewField('ground_truth.label') == label))
        label_dist[label] = count

    print(f"\n{user} ({len(user_dataset)} samples):")
    for label, count in sorted(label_dist.items()):
        print(f"  {label}: {count}")

print("\n" + "="*60)
print("All user datasets created successfully!")
print("="*60)
print("\nDataset names:")
for user in users:
    print(f"  - jaguar_reid_{user.lower()}")

Users in alphabetical order: ['alex', 'antonio', 'juan', 'rami', 'thede', 'uma']

Found 4 unique labels
Total samples: 620
Deleting existing dataset: jaguar_reid_alex
 100% |█████████████████| 101/101 [91.8ms elapsed, 0s remaining, 1.1K samples/s]   


INFO:eta.core.utils: 100% |█████████████████| 101/101 [91.8ms elapsed, 0s remaining, 1.1K samples/s]   



alex's dataset: jaguar_reid_alex
  Samples: 101
  Labels: 4
Deleting existing dataset: jaguar_reid_antonio
 100% |█████████████████| 101/101 [88.3ms elapsed, 0s remaining, 1.1K samples/s]   


INFO:eta.core.utils: 100% |█████████████████| 101/101 [88.3ms elapsed, 0s remaining, 1.1K samples/s]   



antonio's dataset: jaguar_reid_antonio
  Samples: 101
  Labels: 4
Deleting existing dataset: jaguar_reid_juan
 100% |█████████████████| 101/101 [96.3ms elapsed, 0s remaining, 1.0K samples/s]   


...


rami's dataset: jaguar_reid_rami
  Samples: 101
  Labels: 4
Deleting existing dataset: jaguar_reid_thede
 100% |█████████████████| 101/101 [96.8ms elapsed, 0s remaining, 1.0K samples/s]   


INFO:eta.core.utils: 100% |█████████████████| 101/101 [96.8ms elapsed, 0s remaining, 1.0K samples/s]   



thede's dataset: jaguar_reid_thede
  Samples: 101
  Labels: 4
Deleting existing dataset: jaguar_reid_uma
 100% |█████████████████| 115/115 [101.3ms elapsed, 0s remaining, 1.1K samples/s]  


INFO:eta.core.utils: 100% |█████████████████| 115/115 [101.3ms elapsed, 0s remaining, 1.1K samples/s]  



uma's dataset: jaguar_reid_uma
  Samples: 115
  Labels: 4

Dataset Split Summary:

alex (101 samples):
  Alira: 7
  Bororo: 7
  Marcela: 39
  Ousado: 48

antonio (101 samples):
  Alira: 7
  Bororo: 7
  Marcela: 39
  Ousado: 48

juan (101 samples):
  Alira: 7
  Bororo: 7
  Marcela: 39
  Ousado: 48

rami (101 samples):
  Alira: 7
  Bororo: 7
  Marcela: 39
  Ousado: 48

thede (101 samples):
  Alira: 7
  Bororo: 7
  Marcela: 39
  Ousado: 48

uma (115 samples):
  Alira: 10
  Bororo: 12
  Marcela: 42
  Ousado: 51

All user datasets created successfully!

Dataset names:
  - jaguar_reid_alex
  - jaguar_reid_antonio
  - jaguar_reid_juan
  - jaguar_reid_rami
  - jaguar_reid_thede
  - jaguar_reid_uma


In [ ]:
fo.list_datasets()

['jaguar_reid',
 'jaguar_reid_alex',
 'jaguar_reid_antonio',
 'jaguar_reid_juan',
 'jaguar_reid_rami',
 'jaguar_reid_thede',
 'jaguar_reid_uma']

## Start a semantic segmentation annotation job on CVAT

In [ ]:
dataset = fo.load_dataset(f"jaguar_reid_{MY_NAME}")
dataset

Name:        jaguar_reid_antonio
Media type:  image
Num samples: 101
Persistent:  True
Tags:        []
Sample fields:
    id:                fiftyone.core.fields.ObjectIdField
    filepath:          fiftyone.core.fields.StringField
    tags:              fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:          fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:        fiftyone.core.fields.DateTimeField
    last_modified_at:  fiftyone.core.fields.DateTimeField
    ground_truth:      fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Classification)
    image_type:        fiftyone.core.fields.StringField
    original_filename: fiftyone.core.fields.StringField

In [ ]:
session = fo.launch_app(dataset, auto=False)
print(session.url)

Session launched. Run `session.show()` to open the App in a cell output.


INFO:fiftyone.core.session.session:Session launched. Run `session.show()` to open the App in a cell output.


https://5151-m-hm-537h452vexbx-b.asia-east1-0.prod.colab.dev?polling=true


In [ ]:
"""## Start a semantic segmentation annotation job on CVAT

For semantic segmentation, we need to:
1. Set up mask targets (mapping pixel values to class labels)
2. Configure the annotation schema
3. Upload the dataset to CVAT for annotation
"""

# Define mask targets: mapping pixel values to semantic label strings
# We only care about two classes: body and background
mask_targets = {
    0: "background",  # Pixel value 0 = background
    1: "body",        # Pixel value 1 = body
}

# Store mask targets on the dataset
# This allows FiftyOne to automatically use them when creating the annotation task
dataset.default_mask_targets = mask_targets
dataset.save()

print("\nMask targets configured:")
print(f"  0: background")
print(f"  1: body")

"""### Configure and launch the annotation task

We'll create a new field called 'segmentation' to store the semantic segmentation masks.
"""

# Unique identifier for this annotation run
anno_key = "jaguar_segmentation_2"

# Configure the annotation task
# For semantic segmentation with CVAT, we specify:
# - label_field: the field where segmentation masks will be stored
# - label_type: "segmentation" for semantic segmentation
# - mask_targets: the mapping between pixel values and class labels
# - launch_editor: whether to open CVAT in the browser

print(f"\nCreating CVAT annotation task '{anno_key}'...")
print(f"Field: segmentation")
print(f"Type: semantic segmentation")
print(f"Classes: {list(mask_targets.values())}")

results = dataset.annotate(
    anno_key,
    label_field="segmentation",
    label_type="segmentation",
    mask_targets=mask_targets,
    launch_editor=False,  # Set to True to automatically open CVAT
)

# Print information about the created annotation task
print("\nAnnotation task created successfully!")
print(dataset.get_annotation_info(anno_key))

"""### View task information"""

# Get the status of the annotation task
print("\nTask Status:")
results.print_status()

# Get the CVAT task URL
print("\nTo annotate in CVAT, visit the task URL shown above.")
print("Or use the CVAT UI at: https://app.cvat.ai")

"""### After annotation is complete, load the results

Once you've completed the annotation work in CVAT, run the following code
to download and merge the annotations back into your FiftyOne dataset:
"""

# Uncomment the following lines after completing annotation in CVAT:

# # Load annotations from CVAT back into FiftyOne
# dataset.load_annotations(anno_key)
#
# # View the annotated samples
# session = fo.launch_app(dataset)
#
# # Inspect a sample with its segmentation mask
# sample = dataset.first()
# print(sample.segmentation)

"""### Optional: Cleanup after annotation

If you want to delete the CVAT tasks and remove the annotation run record
from your FiftyOne dataset, use:
"""




Mask targets configured:
  0: background
  1: body

Creating CVAT annotation task 'jaguar_segmentation_2'...
Field: segmentation
Type: semantic segmentation
Classes: ['background', 'body']
Computing metadata...


INFO:fiftyone.core.metadata:Computing metadata...


 100% |█████████████████| 101/101 [179.6ms elapsed, 0s remaining, 562.4 samples/s]     


INFO:eta.core.utils: 100% |█████████████████| 101/101 [179.6ms elapsed, 0s remaining, 562.4 samples/s]     


Uploading samples to CVAT...


...

INFO:fiftyone.utils.cvat:	Task 1748725 (FiftyOne_jaguar_reid_antonio):
		Status: annotation
		Assignee: None
		Last updated: 2025-11-04T10:10:58.399741Z
		URL: https://app.cvat.ai/tasks/1748725



		Job 3203837:
			Status: annotation
			Assignee: None
			Reviewer: None



INFO:fiftyone.utils.cvat:		Job 3203837:
			Status: annotation
			Assignee: None
			Reviewer: None




To annotate in CVAT, visit the task URL shown above.
Or use the CVAT UI at: https://app.cvat.ai


'### Optional: Cleanup after annotation\n\nIf you want to delete the CVAT tasks and remove the annotation run record\nfrom your FiftyOne dataset, use:\n'

In [ ]:
# Uncomment to cleanup:
# # Delete tasks from CVAT
results.cleanup()
#
# # Delete run record (not the labels) from FiftyOne
dataset.delete_annotation_run(anno_key)


Deleting tasks...


INFO:fiftyone.utils.cvat:Deleting tasks...


 100% |█████████████████████| 1/1 [1.1s elapsed, 0s remaining, 0.9 samples/s] 


INFO:eta.core.utils: 100% |█████████████████████| 1/1 [1.1s elapsed, 0s remaining, 0.9 samples/s] 
